## Diving into the details of the inclusion implementations.

In [1]:
%load_ext autoreload
%autoreload 2

In [53]:
import numpy as np
from quickrules.data_handling import get_dataset
from pathlib import Path
import fuzzyroughrules.fuzzy_operators as fo
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from fuzzyroughrules.rule_induction_base import RuleGenerator
from fuzzyroughrules.approximations import LowerApproximation, MulticlassMSECVXApproximation, MulticlassMAECVXApproximation
from fuzzyroughrules.operators import ImplicatorInclusion, OWAImplicatorInclusion
from quickrules.data_handling import test_save
from pathlib import Path
from sklearn.metrics import balanced_accuracy_score

In [54]:
name = 'bupa' # 'bupa
nr = 4

In [55]:
x_train, y_train, t_train = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tra",
    get_datatypes=True
)
x_test, y_test = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tst"
)

In [56]:
classes = list(np.unique(np.append(y_test, y_train)))

In [57]:
y_train = np.array([classes.index(label) for label in y_train])
y_test = np.array([classes.index(label) for label in y_test])

In [58]:
model = RuleGenerator(
    approximation=MulticlassMAECVXApproximation(),
    apply_relabelling=True,
    print_changes=True,
    # inclusion_measure=ImplicatorInclusion(),
    inclusion_threshold=0.8,
    with_reducts=True,
    covering_threshold=0.5
)
model.fit(x_train, y_train, _)
balanced_accuracy_score(y_test, model.predict(x_test))


130 labels out of 311 were changed.


0.5

In [52]:
len(model.rules_)

71

In [27]:
if not 0:
    print('yes')

yes


Comparing this to acc when using a higher threshold

In [ ]:
from quickrules.weights import Weights, LinearWeights, TruncatedWeights

model_high = RuleGenerator(
    LowerApproximation(),
    # inclusion_measure=OWAImplicatorInclusion(
    #     weight_function=
    #         LinearWeights()
    # ),
    inclusion_measure=ImplicatorInclusion(),
    inclusion_threshold=1-1e-6,
    with_reducts=True,
    optimise_attribute_order=True
)
model_high.fit(x_train, y_train, t_train)
balanced_accuracy_score(y_test, model_high.predict(x_test))

In [ ]:
high_credibility_predictions = []
for i in range(model_high.n_classes):
    high_credibility_predictions.append([])
for i in model_high.selected_indexes:
    high_credibility_predictions[model_high.y[i]].append(
        fo.lukasiewicz_t_norm(
            triangular_relation(
                x_test_scaled,
                model_high.X_scaled[i],
                model_high.slopes[i],
                model_high.reducts[i]
            ),
            model_high.positive_region[i]
        )
    )

In [ ]:
high_credibility_predictions[6][0]

In [ ]:
for i in range(model_high.n_classes):
    print(len(high_credibility_predictions[i]))